# Commercial Building Energy Forecast
## Electricity Consumption Prediction | COMPLETE REFERENCE

Four-stage pipeline: **prepare data → train two regression models → compare held-back performance → predict new records.** The same fitted preprocessing steps are used for training, evaluation and prediction. Function names and return values are mandatory; do not print inside the functions.

### Imports and constants
Shared across all pipeline functions. Input files live in `data/`; outputs go to `processed_data/`, `artifacts/` and `output/`.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "energy_consumed_kwh"
ID_COL = "reading_id"
NUMERIC_COLS = ["floor_area_sqm", "occupancy_count", "operating_hours", "outdoor_temp_c", "humidity_pct"]
CATEGORICAL_COLS = ["building_code", "hvac_mode", "day_type"]

### Function 1 — Clean and prepare data
**`prepare_data`** — Read CSV, drop duplicate reading IDs, drop invalid timestamps, normalize categories, coerce numerics and median-fill, engineer `reading_hour` and `occupancy_density`, one-hot encode, save and return the DataFrame.

In [ ]:
def prepare_data(input_path, output_path, training=False):
    """Clean the raw CSV, engineer features, one-hot encode, save and return the prepared DataFrame.

    - Read CSV with pandas; drop duplicate reading_id (keep first).
    - Parse reading_ts with errors="coerce"; drop rows with invalid timestamps.
    - Trim spaces + lowercase building_code, hvac_mode, day_type.
    - pd.to_numeric(..., errors="coerce") on numeric columns; fill missing numeric
      features with the median of the CURRENT dataset.
    - Create reading_hour and occupancy_density = occupancy_count / floor_area_sqm
      (avoid division by zero).
    - pd.get_dummies() on building_code, hvac_mode, day_type.
    - Write to output_path without index and return the DataFrame.
    """
    df = pd.read_csv(input_path)
    df = df.drop_duplicates(subset=ID_COL, keep="first").copy()

    timestamps = pd.to_datetime(df["reading_ts"], format="mixed", dayfirst=True, errors="coerce")
    df = df.loc[timestamps.notna()].copy()
    timestamps = timestamps.loc[df.index]

    for col in CATEGORICAL_COLS:
        df[col] = df[col].astype("string").str.strip().str.lower()

    numeric_cols = NUMERIC_COLS + ([TARGET] if TARGET in df.columns else [])
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    for col in NUMERIC_COLS:
        df[col] = df[col].fillna(df[col].median())

    # Labelled data cannot be used without a target value
    if training and TARGET in df.columns:
        df = df.dropna(subset=[TARGET]).copy()
        timestamps = timestamps.loc[df.index]

    df["reading_hour"] = timestamps.dt.hour
    safe_area = df["floor_area_sqm"].where(df["floor_area_sqm"] > 0)
    df["occupancy_density"] = (df["occupancy_count"] / safe_area).fillna(0.0)

    df = df.drop(columns=["reading_ts"])
    df = pd.get_dummies(df, columns=CATEGORICAL_COLS, dtype=int)

    df.to_csv(output_path, index=False)
    return df

### Function 2 — Train models
**`train_models`** — Fit the median-imputation + StandardScaler preprocessor, train LinearRegression and RandomForestRegressor, and save all three objects with joblib.

In [ ]:
def train_models(processed_train_path, preprocessor_path, linear_model_path, forest_model_path):
    """Fit the preprocessor and both regression models, save them with joblib, return both models.

    - Separate reading_id and energy_consumed_kwh from the predictor columns.
    - Pipeline: median imputation + StandardScaler on the predictor columns.
    - LinearRegression (baseline).
    - RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_leaf=2,
      random_state=42, n_jobs=-1).
    - Save preprocessor + both models; return (linear_model, forest_model).
    """
    df = pd.read_csv(processed_train_path)
    y = df[TARGET]
    X = df.drop(columns=[ID_COL, TARGET])

    preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    X_scaled = preprocessor.fit_transform(X)

    linear_model = LinearRegression()
    linear_model.fit(X_scaled, y)

    forest_model = RandomForestRegressor(
        n_estimators=150, max_depth=10, min_samples_leaf=2,
        random_state=42, n_jobs=-1,
    )
    forest_model.fit(X_scaled, y)

    joblib.dump(preprocessor, preprocessor_path)
    joblib.dump(linear_model, linear_model_path)
    joblib.dump(forest_model, forest_model_path)
    return linear_model, forest_model

### Function 3 — Evaluate and select
**`evaluate_and_select`** — Load saved objects (no refitting), compute MAE / RMSE / R² for both models, rank by RMSE ascending, save the report and return the best model path with the comparison table.

In [ ]:
def evaluate_and_select(test_path, preprocessor_path, linear_model_path, forest_model_path, report_path):
    """Score both saved models on the held-back data and select the lowest-RMSE model.

    - Load test data, preprocessor and both models. Do NOT refit anything.
    - Compute MAE, RMSE, R2 for each model.
    - Comparison DataFrame columns: model_name, mae, rmse, r2 (ordered by rmse ascending).
    - Save the table to report_path.
    - Return (selected_model_path, comparison_df).
    """
    df = pd.read_csv(test_path)
    preprocessor = joblib.load(preprocessor_path)
    models = {
        "linear_regression": (joblib.load(linear_model_path), linear_model_path),
        "random_forest": (joblib.load(forest_model_path), forest_model_path),
    }

    y_true = df[TARGET]
    X = df.drop(columns=[ID_COL, TARGET]).reindex(columns=list(preprocessor.feature_names_in_), fill_value=0)
    X_scaled = preprocessor.transform(X)

    rows = []
    for name, (model, _) in models.items():
        pred = model.predict(X_scaled)
        rows.append({
            "model_name": name,
            "mae": mean_absolute_error(y_true, pred),
            "rmse": float(np.sqrt(mean_squared_error(y_true, pred))),
            "r2": r2_score(y_true, pred),
        })

    comparison = pd.DataFrame(rows).sort_values("rmse", ascending=True).reset_index(drop=True)
    comparison.to_csv(report_path, index=False)

    best_name = comparison.loc[0, "model_name"]
    return models[best_name][1], comparison

### Function 4 — Predict new data
**`predict_energy`** — Prepare the scheduled data, align to training feature order, predict with the selected model, clip negatives to 0, round to 1 decimal and save `reading_id` + `predicted_energy_consumed_kwh`.

In [ ]:
def predict_energy(model_path, preprocessor_path, input_path, processed_path, output_path):
    """Prepare scheduled data, predict with the selected model, write and return predictions.

    - Prepare the scheduled dataset with the same cleaning/feature logic.
    - Align predictor columns to the training feature order (add missing dummies = 0,
      ignore unexpected dummies).
    - Load selected model + fitted preprocessor, transform, predict.
    - Replace negatives with 0, round to 1 decimal place.
    - Write reading_id + predicted_energy_consumed_kwh to output_path (no index),
      preserve input order, return the DataFrame.
    """
    df = prepare_data(input_path, processed_path, training=False)

    preprocessor = joblib.load(preprocessor_path)
    model = joblib.load(model_path)

    feature_order = list(preprocessor.feature_names_in_)
    X = df.reindex(columns=feature_order, fill_value=0)

    # Save the scheduled rows aligned to the training features
    aligned = pd.concat([df[[ID_COL]], X], axis=1)
    aligned.to_csv(processed_path, index=False)

    pred = model.predict(preprocessor.transform(X))
    pred = np.round(np.clip(pred, 0, None), 1)

    result = pd.DataFrame({
        ID_COL: df[ID_COL].to_numpy(),
        "predicted_energy_consumed_kwh": pred,
    })
    result.to_csv(output_path, index=False)
    return result

### Main block
Creates folders, prepares train/test data, trains both models, compares them, selects the lowest-RMSE model and predicts the scheduled readings. Printing happens **only here**, not inside the functions.

In [ ]:
if __name__ == "__main__":
    for folder in ("processed_data", "artifacts", "output"):
        os.makedirs(folder, exist_ok=True)

    train_df = prepare_data("data/building_train.csv", "processed_data/building_train_cleaned.csv", training=True)
    test_df = prepare_data("data/building_test.csv", "processed_data/building_test_cleaned.csv", training=True)

    train_models(
        "processed_data/building_train_cleaned.csv",
        "artifacts/preprocessor.pkl",
        "artifacts/linear_regression.pkl",
        "artifacts/random_forest.pkl",
    )

    selected_model_path, comparison = evaluate_and_select(
        "processed_data/building_test_cleaned.csv",
        "artifacts/preprocessor.pkl",
        "artifacts/linear_regression.pkl",
        "artifacts/random_forest.pkl",
        "output/model_comparison.csv",
    )

    predictions = predict_energy(
        selected_model_path,
        "artifacts/preprocessor.pkl",
        "data/building_predict.csv",
        "processed_data/building_predict_cleaned.csv",
        "output/building_predictions.csv",
    )

    print("Prepared training rows:", train_df.shape)
    print("Prepared test rows:", test_df.shape)
    print("Model comparison:")
    print(comparison.round(3).to_string(index=False))
    print("Selected model:", selected_model_path)
    print("Predictions written:", predictions.shape)
    print(predictions.head(5).to_string(index=False))